In [6]:
from langchain.chat_models import ChatOpenAI
from langchain.schema import HumanMessage, AIMessage, SystemMessage
from langchain.globals import set_llm_cache, set_debug
from langchain.cache import InMemoryCache, SQLiteCache

set_llm_cache(SQLiteCache("movie_info.db"))


chat = ChatOpenAI(temperature=0.1)

messages = [HumanMessage(
                content="Give me information of 10 movies latest released \
                including title, director, main cast, budget, revenue, genre, and brief synopsis. \
                Give  me a short answer."
            )]

res = chat.predict_messages(messages)
print(res.content)


1. Title: Black Widow
   Director: Cate Shortland
   Main Cast: Scarlett Johansson, Florence Pugh, David Harbour
   Budget: $200 million
   Revenue: $379 million
   Genre: Action, Adventure, Sci-Fi
   Synopsis: Natasha Romanoff confronts the darker parts of her ledger when a dangerous conspiracy with ties to her past arises.

2. Title: Jungle Cruise
   Director: Jaume Collet-Serra
   Main Cast: Dwayne Johnson, Emily Blunt, Jack Whitehall
   Budget: $200 million
   Revenue: $215 million
   Genre: Action, Adventure, Comedy
   Synopsis: A riverboat captain takes a scientist and her brother on a mission into a jungle to find the Tree of Life.

3. Title: The Suicide Squad
   Director: James Gunn
   Main Cast: Margot Robbie, Idris Elba, John Cena
   Budget: $185 million
   Revenue: $167 million
   Genre: Action, Adventure, Comedy
   Synopsis: Supervillains Harley Quinn, Bloodsport, Peacemaker, and a collection of nutty cons at Belle Reve prison join the super-secret, super-shady Task Force X

In [11]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.prompts.few_shot import FewShotChatMessagePromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.globals import set_llm_cache, set_debug
from langchain.cache import InMemoryCache, SQLiteCache

set_llm_cache(SQLiteCache("movie_info.db"))


chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[
        StreamingStdOutCallbackHandler(),
    ],
)

examples = [
    {
        "movie_title": "Black Widow",
        "answer": """
            Title: Black Widow
            Director: Cate Shortland
            Main Cast: Scarlett Johansson, Florence Pugh, David Harbour
            Budget: $200 million
            Revenue: $379 million
            Genre: Action, Adventure, Sci-Fi
            Synopsis: Natasha Romanoff confronts the darker parts of her ledger when a dangerous conspiracy with ties to her past arises.
        """,
    },
    {
        "movie_title": "Jungle Cruise",
        "answer": """
            Title: Jungle Cruise
            Director: Jaume Collet-Serra
            Main Cast: Dwayne Johnson, Emily Blunt, Jack Whitehall
            Budget: $200 million
            Revenue: $215 million
            Genre: Action, Adventure, Comedy
            Synopsis: A riverboat captain takes a scientist and her brother on a mission into a jungle to find the Tree of Life.
        """,
    },
    {
        "movie_title": "The Suicide Squad",
        "answer": """
            Title: The Suicide Squad
            Director: James Gunn
            Main Cast: Margot Robbie, Idris Elba, John Cena
            Budget: $185 million
            Revenue: $167 million
            Genre: Action, Adventure, Comedy
            Synopsis: Supervillains Harley Quinn, Bloodsport, Peacemaker, and a collection of nutty cons at Belle Reve prison join the super-secret, super-shady Task Force X.
        """,
    },
]

example_template = '''
    Human: {question}
    AI: {answer}
'''

example_prompt = ChatPromptTemplate.from_messages([
    ('human', 'Tell me about the details of the movie titled {movie_title}.'),
    ('ai', '{answer}')
]) 


example_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

final_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You give short answers for the movie asked.'),
    example_prompt,
    ('human', 'Tell me about the details of the movie titled {movie_title}.')
])

chain = final_prompt | chat
chain.invoke({
    'movie_title': 'Don\'t Breathe 2'
})


            Title: Don't Breathe 2
            Director: Rodo Sayagues
            Main Cast: Stephen Lang, Brendan Sexton III, Madelyn Grace
            Budget: $15 million
            Revenue: $41 million
            Genre: Horror, Thriller
            Synopsis: The Blind Man has been hiding out for years in an isolated cabin and has taken in and raised a young girl orphaned from a house fire. Their quiet existence is shattered when a group of kidnappers show up and take the girl, forcing the Blind Man to leave his safe haven to save her.
        

AIMessageChunk(content="\n            Title: Don't Breathe 2\n            Director: Rodo Sayagues\n            Main Cast: Stephen Lang, Brendan Sexton III, Madelyn Grace\n            Budget: $15 million\n            Revenue: $41 million\n            Genre: Horror, Thriller\n            Synopsis: The Blind Man has been hiding out for years in an isolated cabin and has taken in and raised a young girl orphaned from a house fire. Their quiet existence is shattered when a group of kidnappers show up and take the girl, forcing the Blind Man to leave his safe haven to save her.\n        ")